# Gemma Simplification Evaluation: Jargonaut (G1T5)

**Evaluation criteria:** Does the model improve understandability while preserving contractual meaning?

This section evaluates the Gemma simplification stage on three axes:

| Criteria | Metric | What it answers |
|---|---|---|
| Readability | Flesch–Kincaid Grade (+ Reading Ease) | Is the output easier to read than the source? (Target: ~Grade 8, matching SimpLegal-Pref *Basic*) |
| Simplification quality | SARI | Did the model keep, add and delete the right content compared with the human rewrite? |
| Faithfulness | NLI entailment (both directions) + rule-based critical-element checks | Did the meaning survive: parties, obligations, conditions, exceptions, negation, dates, quantities? |

Each metric has a blind spot on its own:
- FK Grade can be gamed by short, choppy output that drops meaning.
- SARI rewards matching one human reference, so it penalises valid alternative phrasings.
- Faithfulness checks catch dropped negations or changed numbers, but say nothing about readability.

With all metric taken together, this helps to evaluate if the output is *understandable AND meaning-preserving*.

**Systems compared** (so improvements can be attributed to fine-tuning):
1. `copy_source`: returns the legal text unchanged (lower bound).
2. `gemma_zeroshot`: base Gemma with a simplification prompt only.
3. `gemma_finetuned`: our LoRA/QLoRA model on SimpLegal-Pref Basic pairs.
4. `human_reference`: the SimpLegal-Pref Basic rewrite (upper reference).

In [ ]:
!pip install -q textstat evaluate nltk transformers accelerate peft bitsandbytes \
    pandas numpy scipy matplotlib sentencepiece tqdm

In [ ]:
import re, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import textstat
import evaluate
import nltk
from tqdm.auto import tqdm
from scipy import stats

warnings.filterwarnings("ignore")
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ---------- CONFIG: edit these ----------
TEST_PATH      = "simplegal_basic_test.jsonl"  # held-out split; fields: source, reference
BASE_MODEL_ID  = "google/gemma-4-31b"          # <-- replace with the exact HF id you trained from
ADAPTER_PATH   = "outputs/gemma-qlora-adapter" # <-- your fine-tuned LoRA adapter dir
PREDICTIONS_CSV = "predictions.csv"            # cache so generation only runs once
NLI_MODEL_ID   = "MoritzLaurer/DeBERTa-v3-large-mnli-fever-anli-ling-wanli"
MAX_NEW_TOKENS = 384
N_BOOTSTRAP    = 2000

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

## 1. Load the held-out test set

Note: the test pairs must not appear in Gemma's training data. With only ~201 paragraphs in SimpLegal-Pref, the split has to be fixed *before* training (e.g. 80/10/10 with a saved seed). Otherwise every metric below is inflated by memorisation.

The expected file is JSONL with `source` (legal paragraph) and `reference` (checked Basic rewrite).

In [ ]:
test_df = pd.read_json(TEST_PATH, lines=True)[["source", "reference"]].dropna().reset_index(drop=True)
test_df["id"] = test_df.index
print(f"{len(test_df)} test examples")
test_df.head(3)

## 2. Generate predictions (skip if you already have `predictions.csv`)

We generate with **greedy decoding** (`do_sample=False`) so results are reproducible and differences between systems come from the model, not sampling noise. The same prompt template should be used here as in fine-tuning. The zero-shot baseline uses it too, which makes the comparison fair.

In [ ]:
import os
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

PROMPT = (
    "Rewrite the following contract text in plain English at about a Grade 8 reading level. "
    "Preserve all parties, obligations, conditions, exceptions, negations, dates and quantities. "
    "Do not add information.\n\nContract text:\n{source}\n\nPlain English:\n"
)

def load_model(adapter_path=None):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, quantization_config=bnb, device_map="auto")
    if adapter_path:
        model = PeftModel.from_pretrained(model, adapter_path)
    model.eval()
    return tok, model

@torch.no_grad()
def generate_all(tok, model, sources, batch_size=4):
    outs = []
    tok.padding_side = "left"
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    for i in tqdm(range(0, len(sources), batch_size)):
        batch = [PROMPT.format(source=s) for s in sources[i:i+batch_size]]
        enc = tok(batch, return_tensors="pt", padding=True, truncation=True, max_length=2048).to(model.device)
        gen = model.generate(**enc, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        gen = gen[:, enc["input_ids"].shape[1]:]
        outs += [t.strip() for t in tok.batch_decode(gen, skip_special_tokens=True)]
    return outs

if os.path.exists(PREDICTIONS_CSV):
    pred_df = pd.read_csv(PREDICTIONS_CSV)
    print("Loaded cached predictions")
else:
    pred_df = test_df.copy()
    pred_df["copy_source"] = pred_df["source"]

    tok, m = load_model(adapter_path=None)               # zero-shot baseline
    pred_df["gemma_zeroshot"] = generate_all(tok, m, pred_df["source"].tolist())
    del m; torch.cuda.empty_cache()

    tok, m = load_model(adapter_path=ADAPTER_PATH)       # fine-tuned
    pred_df["gemma_finetuned"] = generate_all(tok, m, pred_df["source"].tolist())
    del m; torch.cuda.empty_cache()

    pred_df.to_csv(PREDICTIONS_CSV, index=False)

SYSTEMS = ["copy_source", "gemma_zeroshot", "gemma_finetuned"]
pred_df.head(2)

## 3. Readability: Flesch–Kincaid

**Flesch–Kincaid Grade Level** estimates the US school grade needed to understand the text, from words per sentence and syllables per word. **Flesch Reading Ease** is the same idea on a 0–100 scale (higher score means easier to read).

Interpretation of results:
- Our target is roughly **Grade 8**, matching the *Basic* rewrites.
- The key figure is the **grade reduction** (source grade − output grade), computed per example and so paired.
- Legal paragraphs are often single run-on sentences, so source FK can be extreme (>25). We report medians alongside means.

Note: FK only counts surface features. A model that outputs very short sentences can score "easy" while losing meaning. This is why FK is never reported alone.


In [ ]:
def fk_grade(t):  return textstat.flesch_kincaid_grade(str(t))
def fk_ease(t):   return textstat.flesch_reading_ease(str(t))
def n_words(t):   return len(str(t).split())

cols = SYSTEMS + ["reference"]
read = pd.DataFrame({"id": pred_df["id"]})
read["source_fk"] = pred_df["source"].map(fk_grade)
for c in cols:
    read[f"{c}_fk"]   = pred_df[c].map(fk_grade)
    read[f"{c}_ease"] = pred_df[c].map(fk_ease)
    read[f"{c}_len_ratio"] = pred_df[c].map(n_words) / pred_df["source"].map(n_words)
    read[f"{c}_fk_reduction"] = read["source_fk"] - read[f"{c}_fk"]

summary_read = pd.DataFrame({
    c: {
        "FK grade (mean)":   read[f"{c}_fk"].mean(),
        "FK grade (median)": read[f"{c}_fk"].median(),
        "Reading ease":      read[f"{c}_ease"].mean(),
        "FK reduction":      read[f"{c}_fk_reduction"].mean(),
        "Length ratio (out/src)": read[f"{c}_len_ratio"].mean(),
        "% at or below Grade 8":  (read[f"{c}_fk"] <= 8).mean() * 100,
    } for c in cols
}).T.round(2)
print(f"Source FK grade: mean={read['source_fk'].mean():.2f}, median={read['source_fk'].median():.2f}")
summary_read

## 4. Simplification quality: SARI

**SARI** (System output Against References and against the Input sentence) scores a simplification by comparing what the system **keeps**, **adds** and **deletes** against both the source and the reference(s). It returns a 0–100 score, averaged over those three operations. A higher score means the model’s edits are closer to the human reference’s edits.

Interpretation of results:
- It rewards edits that match the human rewrite, and penalises copying the source unchanged or making unrelated edits.
- With a **single reference** (our case), SARI under-credits valid alternative phrasings. Absolute values will look modest, so read it **comparatively** (fine-tuned vs zero-shot vs copy).
- SARI does **not** check factual correctness. A fluent output that flips "shall not" to "shall" can still score well. That gap is why additional metrics (Section 5) exist.
- We compute SARI per example and bootstrap confidence intervals.

In [ ]:
sari_metric = evaluate.load("sari")

def sari_one(src, pred, ref):
    return sari_metric.compute(sources=[str(src)], predictions=[str(pred)],
                               references=[[str(ref)]])["sari"]

sari = pd.DataFrame({"id": pred_df["id"]})
for c in SYSTEMS:
    sari[f"{c}_sari"] = [sari_one(s, p, r) for s, p, r in
                         zip(pred_df["source"], pred_df[c], pred_df["reference"])]

summary_sari = sari[[f"{c}_sari" for c in SYSTEMS]].agg(["mean", "median", "std"]).T.round(2)
summary_sari

## 5. Faithfulness: preservation of legal meaning

This is the most important part for a legal use case: a plain-English explanation that is *wrong* is worse than none. We use **two complementary automatic checks**, plus a manual review (Section 8), since no single automatic faithfulness metric is reliable on legal text.

### 5a. NLI entailment 
Using a pretrained NLI model:
- **Precision (hallucination check):** does the *source* entail each sentence of the *output*? Low scores suggest added or altered claims.
- **Recall (omission check):** does the *output* entail each sentence of the *source*? Low scores suggest dropped conditions or exceptions.

For each example, we report the **mean** and the **minimum** sentence score. The minimum matters because a single wrong sentence can invalidate a contract explanation.

*Limitation:* general-domain NLI models struggle with long legal sentences and with legitimately reworded content, so this is treated as a screening signal, not ground truth.

In [ ]:
from transformers import AutoModelForSequenceClassification

nli_tok = AutoTokenizer.from_pretrained(NLI_MODEL_ID)
nli_model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL_ID).to(DEVICE).eval()
ENT_IDX = [i for i, l in nli_model.config.id2label.items() if l.lower().startswith("entail")][0]

def split_sents(t):
    return [s.strip() for s in nltk.sent_tokenize(str(t)) if len(s.strip()) > 3]

@torch.no_grad()
def entail_prob(premise, hypothesis):
    enc = nli_tok(premise, hypothesis, return_tensors="pt", truncation="only_first",
                  max_length=512).to(DEVICE)
    return torch.softmax(nli_model(**enc).logits, dim=-1)[0, ENT_IDX].item()

def nli_faithfulness(source, output):
    """Returns (precision_mean, precision_min, recall_mean, recall_min)."""
    out_s, src_s = split_sents(output), split_sents(source)
    if not out_s or not src_s:
        return (0.0, 0.0, 0.0, 0.0)
    prec = [entail_prob(source, s) for s in out_s]   # source => output sentence?
    rec  = [entail_prob(output, s) for s in src_s]   # output => source sentence?
    return (np.mean(prec), np.min(prec), np.mean(rec), np.min(rec))

nli_rows = []
for c in SYSTEMS + ["reference"]:
    for i, (s, o) in enumerate(tqdm(zip(pred_df["source"], pred_df[c]),
                                    total=len(pred_df), desc=c)):
        p_mean, p_min, r_mean, r_min = nli_faithfulness(s, o)
        nli_rows.append(dict(id=i, system=c, nli_prec_mean=p_mean, nli_prec_min=p_min,
                             nli_rec_mean=r_mean, nli_rec_min=r_min))
nli = pd.DataFrame(nli_rows)
nli.groupby("system")[["nli_prec_mean", "nli_prec_min", "nli_rec_mean", "nli_rec_min"]].mean().round(3)

### 5b. Rule-based critical-element checks

The explanation must preserve **parties, obligations, conditions, exceptions, negation, dates and quantities**. These can be checked deterministically with transparent heuristics:

| Check | Rule | Failure means |
|---|---|---|
| Quantities | Every number/amount/percentage in the source appears in the output | A figure was dropped or changed |
| Hallucinated numbers | Output contains numbers not in the source | A figure was invented |
| Dates | Month names/years in the source appear in the output | A date was lost or changed |
| Negation | Source has negation cues but output has none (or vice versa) | Possible meaning inversion |
| Obligation strength | `shall/must` (obligation) vs `may` (permission) | Obligation turned into permission or the reverse |
| Conditions/exceptions | Source has `unless/except/provided that/subject to/if` but output has no conditional cue | Condition or exception dropped |
| Defined terms/parties | Capitalised terms (e.g. "Licensee") from the source appear in the output | A party/defined term was lost |

These are **heuristics**. They cover the common failure modes cheaply but will produce some false alarms (e.g. "thirty (30) days" → "30 days" is normalised, but "one year" → "12 months" is flagged). Use the flags to prioritise manual review, not as a final verdict

In [ ]:
NEG_CUES  = r"\b(not|no|never|neither|nor|without|cannot|can't|won't|none|nothing)\b"
COND_SRC  = r"\b(unless|except|excepting|provided that|provided,? however|subject to|if|only if|in the event|notwithstanding)\b"
COND_OUT  = r"\b(unless|except|only if|if|but|as long as|only|provided|in case|when|however|apart from|other than)\b"
OBLIG     = r"\b(shall|must|will|agrees? to|is required to|are required to|obligated)\b"
PERMIT    = r"\b(may|can|is permitted|are permitted|is allowed|are allowed|entitled)\b"
MONTHS    = r"(january|february|march|april|june|july|august|september|october|november|december)"

# "one" removed: it matches "no one" / "one of the parties" and causes false flags
WORD_NUMS = {"two":2,"three":3,"four":4,"five":5,"six":6,"seven":7,"eight":8,"nine":9,"ten":10,
             "twelve":12,"fifteen":15,"twenty":20,"thirty":30,"forty":40,"sixty":60,"ninety":90,
             "hundred":100}

def extract_numbers(t):
    # Strip clause references (e.g. "Section 5.2") so dropping them isn't flagged as a lost quantity
    t = re.sub(r"\b(section|article|clause|paragraph|exhibit|schedule)s?\s+\d+(\.\d+)*(\([a-z0-9]+\))*",
               " ", str(t).lower())
    nums = set(re.sub(r"[,$]", "", n) for n in re.findall(r"\$?\d[\d,]*\.?\d*%?", t))
    nums |= {str(v) for w, v in WORD_NUMS.items() if re.search(rf"\b{w}\b", t)}
    return {n.rstrip(".") for n in nums}

def extract_years(t):  return set(re.findall(r"\b(?:19|20)\d{2}\b", str(t)))
def extract_months(t):
    t = str(t)
    found = set(re.findall(rf"\b{MONTHS}\b", t.lower()))
    if re.search(r"\bMay\s+\d", t):   # only count "May" when followed by a day/year
        found.add("may")
    return found

def defined_terms(t):
    """Capitalised mid-sentence tokens, a rough proxy for parties/defined terms."""
    toks = re.findall(r"(?<![.!?]\s)(?<!^)\b([A-Z][a-z]{2,})\b", str(t))
    stop = {"The","This","That","These","Those","Agreement","Section","Article","Party","Parties"}
    return {w for w in toks if w not in stop}

def critical_checks(src, out):
    s, o = str(src).lower(), str(out).lower()
    r = {}
    sn, on = extract_numbers(src), extract_numbers(out)
    r["quantity_recall"]   = len(sn & on) / len(sn) if sn else np.nan
    r["hallucinated_nums"] = len(on - sn)
    sd = extract_years(src) | extract_months(src)
    od = extract_years(out) | extract_months(out)
    r["date_recall"] = len(sd & od) / len(sd) if sd else np.nan
    r["negation_flag"] = int(bool(re.search(NEG_CUES, s)) != bool(re.search(NEG_CUES, o)))
    s_ob, s_pe = bool(re.search(OBLIG, s)), bool(re.search(PERMIT, s))
    o_ob, o_pe = bool(re.search(OBLIG, o)), bool(re.search(PERMIT, o))
    r["modality_flag"] = int((s_ob and not o_ob and o_pe) or (s_pe and not s_ob and o_ob and not o_pe))
    r["condition_dropped"] = int(bool(re.search(COND_SRC, s)) and not re.search(COND_OUT, o))
    st = defined_terms(src)
    r["term_recall"] = len({t for t in st if t.lower() in o}) / len(st) if st else np.nan
    return r

crit_rows = []
for c in SYSTEMS + ["reference"]:
    for i, (s, o) in enumerate(zip(pred_df["source"], pred_df[c])):
        crit_rows.append(dict(id=i, system=c, **critical_checks(s, o)))
crit = pd.DataFrame(crit_rows)

# A single "any red flag" indicator for quick comparison
crit["any_flag"] = ((crit.negation_flag == 1) | (crit.modality_flag == 1) |
                    (crit.condition_dropped == 1) | (crit.hallucinated_nums > 0) |
                    (crit.quantity_recall < 1) | (crit.date_recall < 1)).astype(int)

crit.groupby("system")[["quantity_recall", "date_recall", "term_recall", "hallucinated_nums",
                        "negation_flag", "modality_flag", "condition_dropped", "any_flag"]].mean().round(3)

## 6. Combine results, confidence intervals and significance

With ~20–40 test examples, point estimates alone are misleading. We therefore:
1. Merge all per-example metrics into a single table.
2. Report **bootstrap 95% confidence intervals** (resampling examples).
3. Run **paired Wilcoxon signed-rank tests** for fine-tuned vs zero-shot. The comparison is paired because both systems simplify the same paragraphs, and non-parametric because the metrics aren't normally distributed.

In [ ]:
# Merge into one wide per-example table
wide = pred_df[["id", "source", "reference"] + SYSTEMS].copy()
wide = wide.merge(read, on="id").merge(sari, on="id")
for c in SYSTEMS + ["reference"]:
    sub = nli[nli.system == c].set_index("id")[["nli_prec_mean","nli_prec_min","nli_rec_mean","nli_rec_min"]]
    sub.columns = [f"{c}_{x}" for x in sub.columns]
    wide = wide.merge(sub, left_on="id", right_index=True)
    sub = crit[crit.system == c].set_index("id")[["quantity_recall","negation_flag","modality_flag",
                                                  "condition_dropped","hallucinated_nums","any_flag"]]
    sub.columns = [f"{c}_{x}" for x in sub.columns]
    wide = wide.merge(sub, left_on="id", right_index=True)

def boot_ci(x, n=N_BOOTSTRAP, alpha=0.05):
    x = np.asarray(x, dtype=float); x = x[~np.isnan(x)]
    means = [np.random.choice(x, len(x), replace=True).mean() for _ in range(n)]
    return x.mean(), np.percentile(means, 100*alpha/2), np.percentile(means, 100*(1-alpha/2))

metric_map = {
    "FK grade ↓":             "fk",
    "FK reduction ↑":         "fk_reduction",
    "SARI ↑":                 "sari",
    "NLI precision (mean) ↑": "nli_prec_mean",
    "NLI recall (mean) ↑":    "nli_rec_mean",
    "NLI precision (min) ↑":  "nli_prec_min",
    "Any critical flag ↓":    "any_flag",
}
rows = []
for c in SYSTEMS + ["reference"]:
    for label, suffix in metric_map.items():
        col = f"{c}_{suffix}"
        if col in wide:
            m, lo, hi = boot_ci(wide[col])
            rows.append(dict(system=c, metric=label, mean=m, ci_low=lo, ci_high=hi))
results = pd.DataFrame(rows)
results_pivot = results.assign(
    cell=lambda d: d.apply(lambda r: f"{r['mean']:.2f} [{r['ci_low']:.2f}, {r['ci_high']:.2f}]", axis=1)
).pivot(index="metric", columns="system", values="cell")
results_pivot

In [ ]:
def paired_test(a_col, b_col):
    a, b = wide[a_col].values, wide[b_col].values
    mask = ~(np.isnan(a) | np.isnan(b))
    a, b = a[mask], b[mask]
    if np.allclose(a, b): return np.nan, np.nan
    stat, p = stats.wilcoxon(a, b)
    return (a - b).mean(), p

print("Fine-tuned vs zero-shot (paired Wilcoxon):\n")
for label, suffix in metric_map.items():
    d, p = paired_test(f"gemma_finetuned_{suffix}", f"gemma_zeroshot_{suffix}")
    print(f"{label:28s} mean diff = {d:+.3f}   p = {p:.4f}")

## 7. Visualisation
Three views: (1) FK grade by system, with the Grade 8 target and the source level for context; (2) SARI by system; (3) a **readability vs faithfulness scatter** per example, the key diagnostic. The ideal region is the top-left: low grade (easy to read) and high entailment (faithful). Points in the bottom-left are easy to read but unfaithful. These are over-simplified, which is the main failure mode we are guarding against. 

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(17, 4.5))

# 1. FK grade
data = [wide["source_fk"]] + [wide[f"{c}_fk"] for c in SYSTEMS[1:]] + [wide["reference_fk"]]
ax[0].boxplot(data)
ax[0].set_xticklabels(["source", "zero-shot", "fine-tuned", "human ref"])
ax[0].axhline(8, ls="--", c="gray"); ax[0].set_title("Flesch–Kincaid grade (dashed = Grade 8 target)")
ax[0].set_ylabel("Grade level")

# 2. SARI
ax[1].boxplot([wide[f"{c}_sari"] for c in SYSTEMS])
ax[1].set_xticklabels(["copy", "zero-shot", "fine-tuned"])
ax[1].set_title("SARI (higher is better)")

# 3. Readability vs faithfulness
for c, col in [("gemma_zeroshot", "tab:orange"), ("gemma_finetuned", "tab:blue")]:
    ax[2].scatter(wide[f"{c}_fk"], wide[f"{c}_nli_prec_mean"], alpha=.7, label=c, c=col)
ax[2].axvline(8, ls="--", c="gray"); ax[2].axhline(0.5, ls=":", c="gray")
ax[2].set_xlabel("FK grade (lower = easier)"); ax[2].set_ylabel("NLI: source ⇒ output (higher = more faithful)")
ax[2].set_title("Readability vs faithfulness"); ax[2].legend()

plt.tight_layout(); plt.savefig("gemma_eval_summary.png", dpi=200); plt.show()

## 8. Error analysis and manual review sample

Automatic metrics indicate *where to look*; they don't replace reading outputs. We therefore export:
1. **The worst fine-tuned outputs**: lowest NLI precision, plus any with critical flags.
2. **A random sample** of examples for manual faithfulness labelling (e.g. 20 examples × 7 elements), so we can estimate how well the automatic checks agree with a human, and report that agreement honestly.

In the report, categorise each failure (dropped condition, number changed, negation flipped, hallucinated content, over-simplification, verbose/unchanged) and report counts.

In [ ]:
ft = "gemma_finetuned"
flag_cols = [f"{ft}_negation_flag", f"{ft}_modality_flag", f"{ft}_condition_dropped"]

wide["ft_n_flags"] = wide[flag_cols].sum(axis=1) + (wide[f"{ft}_hallucinated_nums"] > 0).astype(int)
worst = wide.sort_values([ "ft_n_flags", f"{ft}_nli_prec_min"], ascending=[False, True]).head(10)

for _, r in worst.head(5).iterrows():
    print(f"=== id {r['id']} | flags={r['ft_n_flags']} | NLI prec min={r[f'{ft}_nli_prec_min']:.2f} "
          f"| FK={r[f'{ft}_fk']:.1f} | SARI={r[f'{ft}_sari']:.1f}")
    print("SOURCE:   ", r["source"][:500])
    print("OUTPUT:   ", r[ft][:500])
    print("REFERENCE:", r["reference"][:500], "\n")

# Manual-annotation sheet (random sample, blind to system)
sample = wide.sample(n=min(20, len(wide)), random_state=SEED)[["id", "source", ft]].copy()
for dim in ["parties", "obligations", "conditions", "exceptions", "negation", "dates_quantities", "added_info"]:
    sample[f"human_{dim}"] = ""      # fill 1-5 or N/A by hand
sample.to_csv("manual_review_sheet.csv", index=False)
wide.to_csv("gemma_eval_per_example.csv", index=False)
results.to_csv("gemma_eval_summary.csv", index=False)
print("Saved: manual_review_sheet.csv, gemma_eval_per_example.csv, gemma_eval_summary.csv")

## 9. Interpreting the results (template for the report)

**Claim we can support if…**

| Finding | Supports the research question if |
|---|---|
| FK reduction | Fine-tuned output is ≈ Grade 8 and significantly below the source |
| SARI | Fine-tuned > zero-shot > copy, with non-overlapping CIs |
| NLI + critical flags | Fine-tuned faithfulness is close to the human reference and no worse than zero-shot |
| Manual review | Human labels broadly agree with the automatic flags |

**Limitations to state explicitly**
- Small test set (≈20–40 paragraphs) → wide confidence intervals.
- SARI uses a single reference and ignores correctness.
- NLI and regex checks are heuristics; they can miss subtle legal errors and flag harmless rewording.
- The evaluation covers the simplification stage in isolation, using **gold evidence** as input. End-to-end quality (when DeBERTa's extracted span is the input) should be evaluated separately as a team, since extraction errors will propagate to Gemma.
- This is not legal advice; the faithfulness checks do not certify legal equivalence.